## Natural Scenes Dataset Voxel Searchlight RDM Correlation

**setup**
- download the Natural Scenes Dataset (500 gigabytes !) [try this **helper](https://github.com/lucas-nunn/visuo_llm_ram_rescue/blob/main/src/nsd_visuo_semantics/utils/download_nsd_visuo_semantics.py)
- set up your [environment variables](../.env.example)
- `cp .env.example .env`
- choose a model, which may entail writing custom code for extracting embeddings and generating RDMs

**background**
- read this [paper](https://www.nature.com/articles/s42256-025-01072-0)
- go through this [repo](https://github.com/lucas-nunn/visuo_llm_ram_rescue)
- this notebook essentially runs 

In [4]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
env_path = next((candidate for candidate in env_candidates if candidate.exists()), None)
if env_path is None:
    raise FileNotFoundError("Missing .env file. Copy .env.example to .env and fill paths.")
load_dotenv(env_path)

MODEL = "all-mpnet-base-v2"
SUBJECT = 2
NUM_SESSIONS = 20

nsd_dir = os.environ["NSD_DATA_PATH"]
assert Path(nsd_dir).exists(), f"NSD_DATA_PATH does not exist: {nsd_dir}"
figures_dir = f"../figures"
save_path = "../results/embeddings/"
models_rdm_dist = "correlation"
base_save_dir = "../results/searchlight"
betas_dir = f"{base_save_dir}/betas"
precompsl_dir = f"{base_save_dir}/searchlights"
saved_embeddings_dir = f"{base_save_dir}/embeddings"
rdms_dir = f'{base_save_dir}/serialised_models_{models_rdm_dist}'

In [7]:
from nsd_visuo_semantics.get_embeddings.get_nsd_sentence_embeddings_simple import get_nsd_sentence_embeddings_simple

get_nsd_sentence_embeddings_simple(MODEL, "../data/ms_coco_nsd_captions_test.pkl", "eh", save_path, True)

GATHERING EMBEDDINGS FOR: all-mpnet-base-v2
 ON: ../data/ms_coco_nsd_captions_test.pkl


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

running sanity check
Running... 99.86%

In [2]:
from nsd_visuo_semantics.utils.nsd_prepare_modelrdms import nsd_prepare_modelrdms

nsd_prepare_modelrdms(MODEL, models_rdm_dist, saved_embeddings_dir, rdms_dir, nsd_dir, "", "", True, n_sessions=NUM_SESSIONS, n_subjects=SUBJECT)

Loaded all-mpnet-base-v2 features from ../results/searchlight/embeddings/nsd_all-mpnet-base-v2_mean_embeddings.pkl
	feature matrix shape: (73000, 768)
		sub: subj01 fetching condition trials in session: 20Creating all-mpnet-base-v2 rdm for subj01
Saving in ../results/searchlight/serialised_models_correlation/all-mpnet-base-v2/subj01_all-mpnet-base-v2_fullrdm.npy
		sub: subj02 fetching condition trials in session: 20Creating all-mpnet-base-v2 rdm for subj02
Saving in ../results/searchlight/serialised_models_correlation/all-mpnet-base-v2/subj02_all-mpnet-base-v2_fullrdm.npy


In [8]:
from nsd_visuo_semantics.searchlight_analyses.nsd_searchlight_main_tf import nsd_searchlight_main_tf

nsd_searchlight_main_tf(MODEL, models_rdm_dist, 
                        nsd_dir, base_save_dir, betas_dir, base_save_dir, 
                        False, subject=SUBJECT, n_sessions=NUM_SESSIONS)

Starting main searchlight computations for all-mpnet-base-v2
Loading serialised model rdms from ../results/searchlight/serialised_models_correlation/all-mpnet-base-v2
	the output files will be stored in ../results/searchlight/searchlight_respectedsampling_correlation/subj02/all-mpnet-base-v2/corr_vols_correlation..
	looking for saved samples in ../results/searchlight/searchlight_respectedsampling_correlation/subj02/saved_sampling..
	loading brain mask
	loading pre-computed searchlight
		sub: subj02 fetching condition trials in session: 20Loading 100x100 sample choices from ../results/searchlight/searchlight_respectedsampling_correlation/subj02/saved_sampling/subj02_nsd-allsubstim_sampling.npy
loading betas for subj02
loading betas for subj02



	Found existing file at ../results/searchlight/searchlight_respectedsampling_correlation/subj02/all-mpnet-base-v2/corr_vols_correlation/subj02_nsd-all-mpnet-base-v2_func1pt8mm_sample-0.npy, skipping...



	Found existing file at ../results/searc

In [7]:
from nsd_visuo_semantics.searchlight_analyses.nsd_project_fsaverage import nsd_project_fsaverage

nsd_project_fsaverage([MODEL], models_rdm_dist, nsd_dir, base_save_dir)

	reading in ../results/searchlight/searchlight_respectedsampling_correlation/subj01/all-mpnet-base-v2/corr_vols_correlation/subj01_nsd-all-mpnet-base-v2_func1pt8mm_sample-0.npy

reading model fit samples for all-mpnet-base-v2


samples: 100%|##########| 4/4 [00:00<00:00, 177.97it/s]

projecting to fsaverage

	projecting model 0 out of 1 models
		../results/searchlight/searchlight_respectedsampling_correlation/subj01/all-mpnet-base-v2/all-mpnet-base-v2_correlation_fsaverage/lh.subj01-model-1-surf.npy already exists, skipping
	projecting t-values for model 0 out of 1 models
		../results/searchlight/searchlight_respectedsampling_correlation/subj01/all-mpnet-base-v2/all-mpnet-base-v2_correlation_fsaverage/lh.subj01-model-1-surf-tvals.npy already exists, skipping



/home/chuddy/dev/MCNB/PSM-NeuroAI-Final/.venv/lib/python3.10/site-packages/nsd_visuo_semantics/searchlight_analyses/nsd_project_fsaverage.py:121: RuntimeWarning: divide by zero encountered in divide
  t_brain = np.nanmean(brain_vols, axis=0)/(np.nanstd(brain_vols, axis=0)/np.sqrt(n_samples))
/home/chuddy/dev/MCNB/PSM-NeuroAI-Final/.venv/lib/python3.10/site-packages/nsd_visuo_semantics/searchlight_analyses/nsd_project_fsaverage.py:121: RuntimeWarning: invalid value encountered in divide
  t_brain = np.nanmean(brain_vols, axis=0)/(np.nanstd(brain_vols, axis=0)/np.sqrt(n_samples))


In [6]:
import cortex
from pathlib import Path

filestore = Path(cortex.db.filestore)
if not (filestore / "fsaverage" / "surfaces" / "wm_lh.gii").exists():
    raise RuntimeError(
        "Pycortex fsaverage surfaces not found. "
        "Run once: python scripts/setup_pycortex.py"
    )

In [5]:
from nsd_visuo_semantics.utils.py_plot_brain_utils import pyplot_brains_from_models_list

pyplot_brains_from_models_list(
    [MODEL],
    [MODEL],
    f"{base_save_dir}/searchlight_respectedsampling_correlation",
    layer="last",
    contrast_layer="same",
    contrast_same_model=False,
    save_type="png",
    figpath=figures_dir,
    plot_indiv_sub=True,
    plot_subj_avg=False,
    roi_overlay="streams",
    nsd_dir=nsd_dir,
    roi_linecolor="black",
    roi_linewidth=0.8,
)